# Scaling-Law Analysis: Physical vs. Effective Distance in Suicide Contagion

**Question:** does excess risk in neighboring ZCTAs after an index suicide decay with physical distance (km), or with a density/mobility-adjusted "effective distance"? Framework borrows the effective-distance concept from Brockmann & Helbing (*Science*, 2013) — originally used to show air-travel-mediated epidemic spread looks regular once geographic distance is replaced by mobility-flow-weighted distance. Adapted here to social contagion using SafeGraph mobility flow instead of air traffic.

**Design:** event-centered decay-curve estimation, pooled across all index cases — not cluster detection.

**You'll need three inputs:**
- `cases`: one row per suicide — `zcta`, `lat`, `lon`, `date` (age optional, for later subgroup work)
- `population`: `zcta`, `year`, `population` (you already have this harmonized from the SaTScan work)
- `flows`: SafeGraph mobility flow between ZCTAs — schema varies by exactly which SafeGraph product you have access to (Patterns vs. Neighborhood Patterns). Below assumes a table of `origin_zcta`, `dest_zcta`, `flow_count` (device counts or visits between home ZCTA and destination ZCTA, summed over your study period). Adjust the aggregation step to your actual columns — the effective-distance math downstream doesn't care how you got there.

---

## Practical notes before you start

- **Prototype every phase on a sample or a single state first.** The exposure-table build (Phase 2) is the expensive step — confirm correctness and runtime on ~2,000 cases before scaling to the full dataset.
- **SafeGraph schema will not match the placeholder column names above.** The math in Phase 1b is the part to preserve exactly; the aggregation step that produces `origin_zcta, dest_zcta, flow_count` needs to be adapted to whatever fields your specific SafeGraph product provides.
- **Check effective-distance coverage in rural ZCTAs before trusting it there** — thin mobility panel data in low-density areas is a real risk, and it's exactly your rural stratum where getting this right matters most for the headline question.

## Phase 0 — Setup

In [ ]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.sparse.csgraph import dijkstra
from sklearn.neighbors import BallTree, NearestNeighbors
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

EARTH_RADIUS_KM = 6371.0088

cases = pd.read_csv("suicide_cases.csv", parse_dates=["date"])
population = pd.read_csv("zcta_population.csv")  # zcta, year, population
flows = pd.read_csv("safegraph_flows.csv")        # origin_zcta, dest_zcta, flow_count -- adjust to your schema

cases = cases.dropna(subset=["zcta", "lat", "lon", "date"]).reset_index(drop=True)
print(f"{len(cases):,} cases, {cases['zcta'].nunique():,} distinct ZCTAs")

## Phase 1 — Build two distance metrics between ZCTAs

### 1a. Physical distance — nearest-neighbor lookup
You don't need a full distance matrix (33k×33k won't fit in memory comfortably at 10GB+). Build a queryable index instead.

In [ ]:
zcta_coords = cases.drop_duplicates("zcta")[["zcta", "lat", "lon"]].reset_index(drop=True)
zcta_id_to_idx = {z: i for i, z in enumerate(zcta_coords["zcta"])}
coords_rad = np.radians(zcta_coords[["lat", "lon"]].values)

physical_tree = BallTree(coords_rad, metric="haversine")

def physical_neighbors(zcta, k=30):
    """Return (zcta_ids, distance_km) for the k nearest ZCTAs by physical distance."""
    i = zcta_id_to_idx[zcta]
    dist_rad, idx = physical_tree.query(coords_rad[i:i+1], k=k+1)  # +1 to drop self
    dist_km = dist_rad[0][1:] * EARTH_RADIUS_KM
    neighbor_zctas = zcta_coords["zcta"].values[idx[0][1:]]
    return neighbor_zctas, dist_km

### 1b. Effective distance — from SafeGraph flow, following Brockmann & Helbing

Core idea: convert flow into a directed probability, then effective distance between adjacent (directly-connected) ZCTAs is `1 - ln(P_ij)`. For ZCTAs with no direct flow, effective distance is the shortest path through the flow network (a strong indirect connection through an intermediate hub still counts as "close").

In [ ]:
zctas_all = sorted(set(flows["origin_zcta"]) | set(flows["dest_zcta"]))
eff_idx = {z: i for i, z in enumerate(zctas_all)}
n = len(zctas_all)

row = flows["origin_zcta"].map(eff_idx).values
col = flows["dest_zcta"].map(eff_idx).values
weight = flows["flow_count"].astype(float).values

F = sp.csr_matrix((weight, (row, col)), shape=(n, n))

# row-normalize: P_ij = flow(i->j) / total outflow from i
outflow = np.asarray(F.sum(axis=1)).flatten()
outflow_safe = np.where(outflow == 0, 1, outflow)  # avoid div-by-zero; zero-outflow rows stay zero
P = F.multiply(1.0 / outflow_safe[:, None]).tocoo()

# effective distance for direct edges: d = 1 - ln(P), clipped to avoid log(0)
p_clipped = np.clip(P.data, 1e-12, 1.0)
d_direct = 1 - np.log(p_clipped)
D_direct = sp.csr_matrix((d_direct, (P.row, P.col)), shape=(n, n))

# multi-hop effective distance = shortest path through the flow network.
# Only compute FROM zctas that actually have cases -- full all-pairs is unnecessary and expensive.
case_zctas = [z for z in cases["zcta"].unique() if z in eff_idx]
source_indices = [eff_idx[z] for z in case_zctas]

eff_dist_matrix = dijkstra(D_direct, directed=True, indices=source_indices)
# eff_dist_matrix[row i, col j] = effective distance from case_zctas[i] to zctas_all[j]

def effective_neighbors(zcta, k=30):
    i = case_zctas.index(zcta)  # slow lookup if called in a loop -- build a dict if reusing heavily
    d = eff_dist_matrix[i]
    order = np.argsort(d)
    order = order[d[order] < np.inf][:k+1]
    neighbor_zctas = np.array(zctas_all)[order]
    dists = d[order]
    mask = neighbor_zctas != zcta
    return neighbor_zctas[mask][:k], dists[mask][:k]

#### Coverage check before #####
# SafeGraph panel coverage is uneven across rural areas. Before trusting effective distance nationally, check what fraction of your case ZCTAs have non-trivial outflow (`outflow > some_threshold`) — if rural coverage is thin, effective distance will be noisiest exactly where you need it most (your low-density stratum). Worth reporting this coverage rate as a caveat regardless of what you find.


coverage = pd.Series(outflow, index=zctas_all).reindex(case_zctas)
print(f"{(coverage > 10).mean():.1%} of case ZCTAs have >10 total outbound flow observations")

### 1c. Fallback: rank distance (no SafeGraph dependency)

Keep this as a robustness check even if SafeGraph coverage looks good — it's a cheap sanity check for whether SafeGraph is adding real signal over simple density-adjusted rank.

In [ ]:
def rank_neighbors(zcta, k=30):
    zctas_only, dist_km = physical_neighbors(zcta, k=k)
    return zctas_only, np.arange(1, len(zctas_only) + 1)  # rank = 1,2,3...k, ignoring actual km

## Phase 2 — Build the event-centered exposure table

Precompute a ZCTA × week case-count lookup once, then reuse it for every index case instead of re-scanning `cases` per event.

In [ ]:
cases["week"] = cases["date"].dt.to_period("W").dt.start_time
weekly_counts = cases.groupby(["zcta", "week"]).size().rename("n_cases")
weekly_counts = weekly_counts.reindex(
    pd.MultiIndex.from_product(
        [cases["zcta"].unique(), pd.date_range(cases["date"].min(), cases["date"].max(), freq="W")],
        names=["zcta", "week"]
    ), fill_value=0
)
count_lookup = weekly_counts.to_dict()  # (zcta, week_timestamp) -> count

# expected weekly count per zcta from population (indirect standardization against national rate)
national_weekly_rate = len(cases) / weekly_counts.index.get_level_values("week").nunique() / population["population"].sum()
pop_by_zcta_year = population.set_index(["zcta", "year"])["population"]

def expected_weekly(zcta, week):
    try:
        pop = pop_by_zcta_year.loc[(zcta, week.year)]
    except KeyError:
        return np.nan
    return pop * national_weekly_rate


#For each index case, pull observed/expected counts for its neighbors at each rank, for a grid of follow-up windows, under **both** distance definitions:


TIME_WINDOWS_WEEKS = [1, 2, 4, 8, 12]
K_NEIGHBORS = 30

def build_exposure_rows(neighbor_fn, distance_label):
    rows = []
    for _, case in cases.iterrows():
        neighbor_zctas, distances = neighbor_fn(case["zcta"], k=K_NEIGHBORS)
        for rank, (nz, d) in enumerate(zip(neighbor_zctas, distances), start=1):
            for T in TIME_WINDOWS_WEEKS:
                window_weeks = pd.date_range(case["week"], periods=T, freq="W")
                obs = sum(count_lookup.get((nz, w), 0) for w in window_weeks)
                exp = sum((expected_weekly(nz, w) or 0) for w in window_weeks)
                rows.append({
                    "distance_type": distance_label,
                    "rank": rank,
                    "distance_value": d,
                    "time_window_weeks": T,
                    "observed": obs,
                    "expected": exp,
                    "source_zcta": case["zcta"],
                })
    return pd.DataFrame(rows)

# NOTE: this loops per-case per-neighbor per-window -- fine to prototype on a sample first.
sample_cases = cases.sample(2000, random_state=0)
exposure_physical = build_exposure_rows(physical_neighbors, "physical")
# exposure_effective = build_exposure_rows(effective_neighbors, "effective")  # once coverage check passes

# Before running this on the full ~600k+ cases: prototype on a few thousand sampled cases first (as above) to confirm the pipeline and check runtime, then scale up. 
# If it's too slow at full scale, the fix is to vectorize the inner loop with numpy array lookups instead of a Python-level dict/loop — flag this as a known optimization step, not a redesign.

## Phase 3 — Stratify by density, fit decay curves, test which metric aligns

Attach a density stratum to each case based on its **source** ZCTA's population density:

In [ ]:
density = population.groupby("zcta")["population"].mean()  # or bring in area to get true density
density_quartile = pd.qcut(density, 4, labels=["rural", "low_density_metro", "metro", "dense_urban"])

exposure_physical["density_stratum"] = exposure_physical["source_zcta"].map(density_quartile)

#Bin distance (km or rank) and fit a Poisson GLM with an offset for expected counts, testing the stratum × distance interaction:

exposure_physical["distance_bin"] = pd.cut(exposure_physical["distance_value"], bins=10)

exposure_physical = exposure_physical[exposure_physical["expected"] > 0]
exposure_physical["log_expected"] = np.log(exposure_physical["expected"])

model_with_interaction = smf.glm(
    "observed ~ C(distance_bin) * C(density_stratum)",
    data=exposure_physical,
    family=sm.families.Poisson(),
    offset=exposure_physical["log_expected"],
).fit()

model_no_interaction = smf.glm(
    "observed ~ C(distance_bin) + C(density_stratum)",
    data=exposure_physical,
    family=sm.families.Poisson(),
    offset=exposure_physical["log_expected"],
).fit()

lr_stat = 2 * (model_with_interaction.llf - model_no_interaction.llf)
df_diff = model_with_interaction.df_model - model_no_interaction.df_model
from scipy.stats import chi2
p_value = chi2.sf(lr_stat, df_diff)
print(f"LR test for stratum x distance interaction (physical km): p = {p_value:.4f}")
#Repeat identically on `exposure_effective` (or `exposure_rank` as the no-SafeGraph fallback) once built. 
# **The comparison that answers your question:** 
# if the interaction is significant in physical-km space but *not* in effective/rank space, that's evidence for topological scaling. 
# If it's significant in both, or absent in both, that's a real result too — report it as such rather than the one you expected.


## Phase 4 — Visualize

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, (df_exp, label) in zip(axes, [(exposure_physical, "Physical distance (km)")]):
    for stratum in df_exp["density_stratum"].cat.categories:
        sub = df_exp[df_exp["density_stratum"] == stratum]
        curve = sub.groupby("distance_bin").apply(
            lambda g: g["observed"].sum() / g["expected"].sum() if g["expected"].sum() > 0 else np.nan
        )
        ax.plot(range(len(curve)), curve.values, marker="o", label=stratum)
    ax.set_title(label)
    ax.set_xlabel("distance bin")
    ax.set_ylabel("Observed / Expected")
    ax.axhline(1.0, color="gray", linestyle="--")
    ax.legend()

plt.tight_layout()
plt.savefig("decay_curves_comparison.png", dpi=150)

Do the same for the effective/rank-distance version — visually, "topological wins" looks like the strata curves overlapping in one panel and fanning out in the other.
## Phase 5 — Derived output: quantify SaTScan's censoring (secondary, not a new pipeline)

In [ ]:
# Once you have the fitted physical-km decay curve per stratum, find where it returns to ~1 (RR flattens to baseline) and compare that to whatever radius your SaTScan runs used:
for stratum in exposure_physical["density_stratum"].cat.categories:
    sub = exposure_physical[exposure_physical["density_stratum"] == stratum]
    curve = sub.groupby("distance_bin").apply(
        lambda g: g["observed"].sum() / g["expected"].sum() if g["expected"].sum() > 0 else np.nan
    )
    # first bin where RR drops to within 5% of 1.0 -- crude but reportable
    baseline_bins = curve[abs(curve - 1.0) < 0.05]
    print(stratum, "RR returns to baseline around bin:", baseline_bins.index.min() if len(baseline_bins) else "never in range")

Compare this per-stratum "true" decay range against your fixed 150km/15-month (or whatever you land on) SaTScan cap to get a direct, reportable censoring estimate — this is your answer to the "how much does the fixed radius artificially censor low-density detection" question, without a separate analysis.